# B3: تقييم دقة الترانسولفر المدرَّب مقابل FEM الحقيقي

**السياق**: التدريب (`B3_Transolver_Training.ipynb`) بيصغّر طاقة الشكل المرن بس (Deep Energy Method) -- **ما بيشوف ولا إزاحة FEM حقيقية أبدًا**. يعني إنه الخسارة ضلت مستقرة طول 2000 تكرار (النتيجة الناجحة يلي وصلتنا) بيثبت بس إنه التحسين (optimization) ما انفجر -- **مش** إنه الحقل يلي اتعلمته الشبكة قريب من الحل الحقيقي. هاد الدفتر هو **التحقق الفعلي** يلي كان ناقص.

**شو بيعمل بالضبط**:
1. بيحمّل الـcheckpoint المدرَّب (`checkpoint_2000.pt`، التشغيلة الثانية الناجحة بعد إصلاح `OUTPUT_SCALE`).
2. بيحمّل الـ100 عينة FEM الحقيقية (`dataset.h5`) -- هاي العينات **ما استخدمت أبدًا بالتدريب**، كل تكرار تدريب كان عم يولّد عينة عشوائية جديدة بنفس التوزيع، فهاي مقارنة حقيقية بيانات لم تُرَ من قبل (held-out).
3. بيحسب **الخطأ النسبي L2** (relative L2 error) لكل مركبة إزاحة (ux, uy, uz) ومجتمعة، بنفس منهجية B1/B2 بالضبط (`evaluate_dataset_hyperelastic_Q4`).
4. بيقيس **زمن الاستدلال** (inference latency) للشبكة المدرَّبة مقابل زمن حل FEM الحقيقي المسجّل بنفس الداتاسيت، لمقارنة سرعة مباشرة.

**تحذير مهم**: لسا ما في أي رقم دقة حقيقي لهاد الـcheckpoint -- منحن ندخل هالخلية وإحنا **ما منعرف** إذا الدقة كويسة ولا لأ. إذا الخطأ النسبي طلع كبير، هاد مش فشل بالكود، هاد معناه لازم نعيد ضبط (hyperparameters) أو نزود عدد التكرارات ونعيد التدريب -- قرار حقيقي بناءً على أرقام حقيقية، مش افتراض.

**اتفحص محليًا على CPU قبل هيك**: تشغيلة كاملة صغيرة (داتاسيت FEM حقيقي بـ4 عينات + موديل toy مدرَّب 20 تكرار) اشتغلت من غير أخطاء ورجّعت أرقام منطقية (خطأ نسبي مختلف باختلاف المركبة، زمن استدلال بالميلي ثانية) -- دليل إنه الأسلاك (wiring) صحيحة قبل ما نجربها على الـcheckpoint الحقيقي.


In [ ]:
# =====================================================================
#  CELL -- B3 Transolver checkpoint evaluation: the missing accuracy
#  check. Training (B3_Transolver_Training.ipynb) only minimizes the
#  Deep Energy Method loss (Pi = U) -- it never sees a single labeled
#  FEM displacement, so a healthy, stable loss curve proves the
#  optimization did not diverge, NOT that the learned field is close to
#  the true solution. This cell runs the trained checkpoint on the 100
#  real FEM samples (dataset.h5, generated separately, never used as a
#  training label) and computes the actual relative-L2 accuracy, plus
#  an inference-latency vs FEM-solve-time comparison -- matching B1/B2's
#  own evaluate_dataset_hyperelastic_Q4 / benchmark_inference_latency_Q4
#  methodology, extended to B3's 3 displacement components.
# =====================================================================
import os
os.environ['JAX_PLATFORMS'] = 'cpu'

import subprocess
import sys


def run(cmd):
    print('$', ' '.join(str(c) for c in cmd), flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    p.wait()
    if p.returncode != 0:
        raise subprocess.CalledProcessError(p.returncode, cmd)


from google.colab import drive
drive.mount('/content/drive')

REPO = '/content/OMAR'
if not os.path.isdir(REPO):
    run(['git', 'clone', '-b', 'claude/claude-code-question-d307wp',
         'https://github.com/SUHIBAMRO/OMAR.git', REPO])
else:
    run(['git', '-C', REPO, 'fetch', 'origin', 'claude/claude-code-question-d307wp'])
    run(['git', '-C', REPO, 'checkout', 'claude/claude-code-question-d307wp'])
    run(['git', '-C', REPO, 'reset', '--hard', 'origin/claude/claude-code-question-d307wp'])

run([sys.executable, '-m', 'pip', 'install', '-q', 'pyvista<0.49'])
run([sys.executable, '-m', 'pip', 'install', '-q', 'torch-fem'])

WORK = f'{REPO}/Practical_Examples'
os.chdir(WORK)
sys.path.insert(0, WORK)

for _mod_name in list(sys.modules):
    if (_mod_name == 'omar_pfem' or _mod_name.startswith('omar_pfem.')
            or _mod_name == 'torchfem' or _mod_name.startswith('torchfem.')
            or _mod_name == 'pyvista' or _mod_name.startswith('pyvista.')):
        del sys.modules[_mod_name]

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (CPU)')

R = '/content/drive/MyDrive/pfem_run'
CHECKPOINT = f'{R}/b3_training/checkpoint_2000.pt'
DATASET = f'{R}/b3_dataset/dataset.h5'
OUT_JSON = f'{R}/b3_training/eval_B3.json'

assert os.path.exists(CHECKPOINT), f'checkpoint not found: {CHECKPOINT}'
assert os.path.exists(DATASET), f'dataset not found: {DATASET}'

sys.argv = [
    'evaluate_B3.py',
    '--checkpoint', CHECKPOINT,
    '--dataset', DATASET,
    '--out_json', OUT_JSON,
]
from omar_pfem.evaluate_B3 import main
main()

print('\nDone.')
